# 2. SIFT (Scale-Invariant Feature Transform) Pipeline
### Category Identification (BoVW) & Pairwise Object Detection (Homography)

This notebook addresses and fixes the bugs and limitations in `mine/sift.ipynb`:
1. **Critical Bug Fixed**: In `mine/sift.ipynb` Cell 9, test descriptors were appended to `train_kps` (`train_kps.append(item_kp)`), corrupting the training array and leaving `test_kps` empty.
2. **Re-assignment Bug Fixed**: In Cell 7, `bf.knnMatch` result was not stored, causing the loop to use obsolete matches.
3. **Bag of Visual Words (BoVW)**: Brute-force 1-NN pairwise template matching across 500 images is slow and fails for intra-class variation. We implement **BoVW** (visual codebook clustering + SVM) for robust category identification.
4. **Spatial Object Detection**: For detecting specific object instances, we implement **Lowe's Ratio Test + RANSAC Homography + Bounding Box Projection**.



In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix

from dataset import load_split
from sift_pipeline import SIFTBoVWPipeline, SIFTFeatureMatcher

def show_rgb(img, title=None):
    if len(img.shape) == 3:
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    else:
        img_rgb = img
    plt.figure(figsize=(7, 7))
    plt.imshow(img_rgb)
    plt.axis('off')
    if title:
        plt.title(title, fontsize=12)
    plt.show()



## Step 1: Load Data & Visualize SIFT Keypoints


In [ ]:
train_imgs, train_lbls, classes, _ = load_split("train")
test_imgs, test_lbls, _, _ = load_split("test")

sift = cv2.SIFT_create(nfeatures=500)
sample_img = train_imgs[0]
gray = cv2.cvtColor(sample_img, cv2.COLOR_BGR2GRAY)
kps, descs = sift.detectAndCompute(gray, None)

img_kps = cv2.drawKeypoints(sample_img, kps, None, flags=cv2.DRAW_MATCHES_FLAGS_DRAW_RICH_KEYPOINTS)
show_rgb(img_kps, title=f"SIFT Rich Keypoints (Detected: {len(kps)})")



## Step 2: Method A - Bag of Visual Words (BoVW) for Object Identification
1. Extract SIFT descriptors across the training set.
2. Cluster descriptors using KMeans into a visual codebook (e.g. 150 visual words).
3. Build normalized visual word frequency histograms.
4. Train an SVM classifier.



In [ ]:
bovw_pipeline = SIFTBoVWPipeline(nfeatures=400, vocab_size=150)
print("Training SIFT BoVW Pipeline...")
train_info = bovw_pipeline.train(train_imgs, train_lbls, C=1.0)
print(f"Trained visual codebook and SVM in {train_info['train_time_sec']:.2f} seconds!")



## Step 3: Evaluate SIFT BoVW on Test Set


In [ ]:
eval_res = bovw_pipeline.evaluate(test_imgs, test_lbls)
print(f"SIFT BoVW Test Accuracy: {eval_res['accuracy'] * 100:.2f}%")
print(f"Average Inference Latency: {eval_res['avg_time_per_image_ms']:.2f} ms/image
")
print(classification_report(test_lbls, eval_res['predictions'], target_names=bovw_pipeline.classes_))

# Plot confusion matrix
cm = np.array(eval_res["confusion_matrix"])
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(cm, cmap="Purples")
ax.set_xticks(range(len(classes)))
ax.set_yticks(range(len(classes)))
ax.set_xticklabels(classes)
ax.set_yticklabels(classes)
plt.xlabel("Predicted Class")
plt.ylabel("True Class")
plt.title("SIFT BoVW Confusion Matrix")
for i in range(len(classes)):
    for j in range(len(classes)):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", color="red" if i != j else "black")
plt.colorbar(im)
plt.show()



## Step 4: Method B - Pairwise Feature Matching & Homography Object Detection
When detecting a specific template or tracking an object across frames:
1. Match keypoints with `cv2.BFMatcher(cv2.NORM_L2)`.
2. Apply Lowe's ratio test (`m.distance < 0.75 * n.distance`).
3. Compute geometric transformation using RANSAC Homography (`cv2.findHomography`).
4. Project bounding box corners (`cv2.perspectiveTransform`) to localize the object!



In [ ]:
matcher = SIFTFeatureMatcher(nfeatures=500, ratio_thresh=0.75)

# Test matching between two consecutive video frames from train/people
f0 = cv2.imread("../data/train/people/02022_0954.MOV_Bielsko-Biała_Pride_frame_000.jpg")
f1 = cv2.imread("../data/train/people/02022_0954.MOV_Bielsko-Biała_Pride_frame_001.jpg")

loc_res = matcher.localize_object(f0, f1, min_inliers=10)
print(f"Matches: {loc_res['num_matches']} | Verified Inliers: {loc_res['num_inliers']} | Object Detected: {loc_res['detected']}")

# Draw matched keypoints and projected bounding box
if loc_res['detected']:
    scene_annotated = f1.copy()
    pts = np.int32(loc_res['dst_corners']).reshape(-1, 1, 2)
    cv2.polylines(scene_annotated, [pts], True, (0, 255, 0), 4, cv2.LINE_AA)
    
    matches_img = cv2.drawMatches(
        f0, loc_res['kp_template'],
        scene_annotated, loc_res['kp_scene'],
        loc_res['good_matches'][:30], None,
        flags=cv2.DrawMatchesFlags_NOT_DRAW_SINGLE_POINTS
    )
    show_rgb(matches_img, title=f"SIFT Keypoint Matching & Object Localization ({loc_res['num_inliers']} Inliers)")



## Step 5: Save Model


In [ ]:
bovw_pipeline.save_model("sift_bovw_model.pkl")
sz = Path("sift_bovw_model.pkl").stat().st_size / (1024 * 1024)
print(f"Saved SIFT Model to sift_bovw_model.pkl ({sz:.2f} MB)")

